
## Folder 04 - File 01: Silver Customers:


**Transformations:**

- Read bronze_customers.
- Trim spaces from text columns.
- Convert email addresses to lowercase.
- Convert created_at to a date.
- Remove duplicate customer_id records.
- Quarantine records with invalid or missing customer IDs or dates.
- Write valid records to silver_customers.

In [0]:

# ============================================================
# E-COMMERCE LAKEHOUSE
# SILVER CUSTOMERS TRANSFORMATION
# ============================================================


# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


# ------------------------------------------------------------
# 2. PROJECT CONFIGURATION
# ------------------------------------------------------------

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_customers"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_customers"
)

print("Project configuration completed.")


# ------------------------------------------------------------
# 3. CREATE SILVER SCHEMA
# ------------------------------------------------------------

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


# ------------------------------------------------------------
# 4. READ BRONZE CUSTOMERS
# ------------------------------------------------------------

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}"
    )

bronze_df = spark.table(BRONZE_TABLE)

bronze_total = bronze_df.count()

print("BRONZE CUSTOMER RECORDS:", bronze_total)

display(bronze_df.limit(10))


# ------------------------------------------------------------
# 5. STANDARDIZE CUSTOMER COLUMNS
# ------------------------------------------------------------

cleaned_df = (
    bronze_df

    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id").cast("string"))
    )

    .withColumn(
        "first_name",
        F.trim(F.col("first_name").cast("string"))
    )

    .withColumn(
        "last_name",
        F.trim(F.col("last_name").cast("string"))
    )

    .withColumn(
        "email",
        F.lower(F.trim(F.col("email").cast("string")))
    )

    .withColumn(
        "phone",
        F.trim(F.col("phone").cast("string"))
    )

    .withColumn(
        "city",
        F.trim(F.col("city").cast("string"))
    )

    .withColumn(
        "state",
        F.trim(F.col("state").cast("string"))
    )

    .withColumn(
        "created_at",
        F.coalesce(
            F.to_date(F.col("created_at").cast("string")),
            F.to_date(
                F.col("created_at").cast("string"),
                "yyyy-MM-dd"
            )
        )
    )
)

print("Customer standardization completed.")

display(cleaned_df.limit(10))


# ------------------------------------------------------------
# 6. IDENTIFY INVALID CUSTOMER RECORDS
# ------------------------------------------------------------

invalid_condition = (
    F.col("customer_id").isNull()
    | (F.col("customer_id") == "")
    | F.col("created_at").isNull()
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)

    .withColumn(
        "_quarantine_reason",

        F.when(
            F.col("customer_id").isNull()
            | (F.col("customer_id") == ""),
            F.lit("Missing or empty customer_id")
        )

        .when(
            F.col("created_at").isNull(),
            F.lit("Missing or invalid created_at")
        )
    )

    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

valid_df = cleaned_df.filter(
    ~invalid_condition
)

invalid_count = invalid_df.count()
valid_count = valid_df.count()

print("VALID RECORDS:", valid_count)
print("INVALID RECORDS:", invalid_count)

print("QUARANTINE REASONS:")

display(
    invalid_df
    .groupBy("_quarantine_reason")
    .count()
)


# ------------------------------------------------------------
# 7. DEDUPLICATE CUSTOMER RECORDS
# ------------------------------------------------------------

# Keep the latest ingested record.
# Use source file and customer attributes as deterministic
# tie-breakers when ingestion timestamps are identical.

deduplication_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file_modification_time").desc_nulls_last(),
        F.col("_source_file").desc_nulls_last(),
        F.col("created_at").desc_nulls_last(),
        F.col("email").desc_nulls_last(),
        F.col("phone").desc_nulls_last()
    )
)

ranked_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(deduplication_window)
    )
)

duplicate_records_df = (
    ranked_df
    .filter(F.col("_row_number") > 1)
    .drop("_row_number")
    .withColumn(
        "_quarantine_reason",
        F.lit("Duplicate customer_id")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

deduplicated_df = (
    ranked_df
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

deduplicated_count = deduplicated_df.count()
duplicate_count = duplicate_records_df.count()

print("DEDUPLICATED SILVER RECORDS:", deduplicated_count)
print("DUPLICATE RECORDS:", duplicate_count)


# ------------------------------------------------------------
# 8. COMBINE INVALID AND DUPLICATE RECORDS
# ------------------------------------------------------------

# Preserve both types of rejected records in quarantine.

quarantine_df = (
    invalid_df
    .unionByName(
        duplicate_records_df,
        allowMissingColumns=True
    )
)

quarantine_count = quarantine_df.count()

print("TOTAL QUARANTINE RECORDS:", quarantine_count)

display(
    quarantine_df
    .groupBy("_quarantine_reason")
    .count()
)


# ------------------------------------------------------------
# 9. WRITE QUARANTINE DELTA TABLE
# ------------------------------------------------------------

(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print("Quarantine table written:", QUARANTINE_TABLE)


# ------------------------------------------------------------
# 10. WRITE SILVER DELTA TABLE
# ------------------------------------------------------------

(
    deduplicated_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Silver table written:", SILVER_TABLE)


# ------------------------------------------------------------
# 11. RELOAD SAVED TABLES
# ------------------------------------------------------------

silver_df = spark.table(SILVER_TABLE)

quarantine_saved_df = spark.table(
    QUARANTINE_TABLE
)

silver_total = silver_df.count()

quarantine_total = quarantine_saved_df.count()

print("Saved Silver records:", silver_total)
print("Saved quarantine records:", quarantine_total)


# ------------------------------------------------------------
# 12. VALIDATE SILVER CUSTOMER RECORDS
# ------------------------------------------------------------

duplicate_customer_groups = (
    silver_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_customer_ids = (
    silver_df
    .filter(
        F.col("customer_id").isNull()
        | (F.col("customer_id") == "")
    )
    .count()
)

missing_created_dates = (
    silver_df
    .filter(F.col("created_at").isNull())
    .count()
)

print("=" * 60)
print("SILVER CUSTOMER VALIDATION")
print("=" * 60)

print("Silver records:", silver_total)
print("Duplicate customer groups:", duplicate_customer_groups)
print("Missing customer IDs:", missing_customer_ids)
print("Missing creation dates:", missing_created_dates)

if (
    duplicate_customer_groups > 0
    or missing_customer_ids > 0
    or missing_created_dates > 0
):
    raise Exception("Silver customer validation failed.")

print("Silver customer validation passed.")


# ------------------------------------------------------------
# 13. BRONZE TO SILVER RECONCILIATION
# ------------------------------------------------------------

bronze_distinct_ids = (
    bronze_df
    .filter(
        F.col("customer_id").isNotNull()
        & (F.trim(F.col("customer_id")) != "")
    )
    .select("customer_id")
    .distinct()
    .count()
)

silver_distinct_ids = (
    silver_df
    .select("customer_id")
    .distinct()
    .count()
)

bronze_duplicate_groups = (
    bronze_df
    .groupBy("customer_id")
    .count()
    .filter(
        (F.col("count") > 1)
        & F.col("customer_id").isNotNull()
    )
    .count()
)

bronze_missing_from_silver = (
    bronze_df
    .select("customer_id")
    .filter(
        F.col("customer_id").isNotNull()
        & (F.trim(F.col("customer_id")) != "")
    )
    .distinct()
    .join(
        silver_df.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
)

missing_silver_count = bronze_missing_from_silver.count()

print("=" * 60)
print("BRONZE TO SILVER RECONCILIATION")
print("=" * 60)

print("Bronze total rows:", bronze_total)
print("Bronze distinct valid IDs:", bronze_distinct_ids)
print("Bronze duplicate customer groups:", bronze_duplicate_groups)
print("Silver total rows:", silver_total)
print("Silver distinct IDs:", silver_distinct_ids)
print("Unique Bronze IDs missing from Silver:", missing_silver_count)

print("Missing customer IDs:")

display(bronze_missing_from_silver)


# ------------------------------------------------------------
# 14. VERIFY EVERY MISSING ID IS ACCOUNTED FOR
# ------------------------------------------------------------

# A missing ID should be represented in quarantine
# if its record was rejected by validation.
#
# Duplicate records are quarantined too, but their customer ID
# should still exist in Silver if another copy was valid.

unaccounted_missing_ids = (
    bronze_missing_from_silver
    .join(
        quarantine_saved_df.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
)

unaccounted_count = unaccounted_missing_ids.count()

print("Missing IDs without quarantine explanation:", unaccounted_count)

display(unaccounted_missing_ids)

if unaccounted_count > 0:
    print(
        "WARNING: Some missing Bronze IDs are not represented "
        "in quarantine. Investigate before declaring success."
    )


# ------------------------------------------------------------
# 15. FINAL SUMMARY
# ------------------------------------------------------------

print("=" * 60)
print("CUSTOMER SILVER TRANSFORMATION COMPLETED")
print("=" * 60)

print("Bronze table:", BRONZE_TABLE)
print("Silver table:", SILVER_TABLE)
print("Quarantine table:", QUARANTINE_TABLE)

print("Bronze rows:", bronze_total)
print("Valid rows before deduplication:", valid_count)
print("Invalid rows:", invalid_count)
print("Duplicate rows quarantined:", duplicate_count)
print("Silver unique customers:", silver_total)
print("Total quarantine records:", quarantine_total)
print("Unaccounted missing customer IDs:", unaccounted_count)

if (
    silver_total != silver_distinct_ids
    or duplicate_customer_groups != 0
    or missing_customer_ids != 0
    or missing_created_dates != 0
):
    raise Exception("Final Silver customer checks failed.")

print("=" * 60)
print("FINAL SILVER CUSTOMER VALIDATION PASSED")
print("=" * 60)

display(
    silver_df
    .orderBy("customer_id")
    .limit(20)
)
